In [ ]:
# Setup environment for Kaggle
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import math
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

STGCN_DIR = '/tmp/st-gcn'
CTRGCN_DIR = '/tmp/CTR-GCN'

# Clone ST-GCN
if not os.path.exists(STGCN_DIR):
    os.system(f'git clone https://github.com/yysijie/st-gcn.git {STGCN_DIR}')
stgcn_io = Path(f'{STGCN_DIR}/torchlight/torchlight/io.py')
text = stgcn_io.read_text()
if 'weights_only=False' not in text:
    text = text.replace('torch.load(weights_path)', 'torch.load(weights_path, weights_only=False, map_location="cpu")')
    stgcn_io.write_text(text)
os.system(f'pip install -e {STGCN_DIR}/torchlight -q')

# Clone CTR-GCN
if not os.path.exists(CTRGCN_DIR):
    os.system(f'git clone https://github.com/Uason-Chen/CTR-GCN.git {CTRGCN_DIR}')
os.system('pip install -q tensorboardX torchpack fvcore iopath yacs thop')
os.system(f'pip install -e {CTRGCN_DIR}/torchlight -q')
ctrgcn_util = Path(f'{CTRGCN_DIR}/torchlight/torchlight/util.py')
text = ctrgcn_util.read_text()
if 'PaviLogger = None' not in text:
    text = text.replace('from torchpack.runner.hooks import PaviLogger', 'try:\n    from torchpack.runner.hooks import PaviLogger\nexcept ImportError:\n    PaviLogger = None')
    ctrgcn_util.write_text(text)

# Download Pretrained Weights
os.makedirs(f'{STGCN_DIR}/models', exist_ok=True)
if not os.path.exists(f'{STGCN_DIR}/models/st_gcn.ntu-xsub.pt'):
    os.system('pip install gdown -q')
    os.system(f'gdown 18pcNj4Bu4Ub7S3YJSsRNJ45XxA4GyaYG -O {STGCN_DIR}/models/st_gcn.ntu-xsub.pt')

ctrgcn_pretrained_dir = f'{CTRGCN_DIR}/pretrained_model/NTU60_Xsub/CTRGCN_joint_89.9'
os.makedirs(ctrgcn_pretrained_dir, exist_ok=True)
if not os.path.exists(f'{ctrgcn_pretrained_dir}/runs-60-37560.pt'):
    os.system(f'gdown 1eVlsxaODkJ6Zhhwfauf7Q142tCzK1FvC -O {ctrgcn_pretrained_dir}/runs-60-37560.pt')

sys.path.insert(0, STGCN_DIR)
sys.path.insert(0, CTRGCN_DIR)
from net.st_gcn import Model as STGCN_Model
from model.ctrgcn import Model as CTRGCN_Model

print("Setup complete. Pretrained weights downloaded.")

Using device: cuda


Cloning into '/tmp/st-gcn'...
Cloning into '/tmp/CTR-GCN'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.3/296.3 kB 14.0 MB/s eta 0:00:00


Downloading...
From: https://drive.google.com/uc?id=18pcNj4Bu4Ub7S3YJSsRNJ45XxA4GyaYG
To: /tmp/st-gcn/models/st_gcn.ntu-xsub.pt
100%|██████████| 12.5M/12.5M [00:00<00:00, 89.5MB/s]


Setup complete. Pretrained weights downloaded.


Downloading...
From: https://drive.google.com/uc?id=1eVlsxaODkJ6Zhhwfauf7Q142tCzK1FvC
To: /tmp/CTR-GCN/pretrained_model/NTU60_Xsub/CTRGCN_joint_89.9/runs-60-37560.pt
100%|██████████| 5.95M/5.95M [00:00<00:00, 56.8MB/s]


In [ ]:
# Load full 2940 dataset and create 5-fold splits
import pickle
from sklearn.model_selection import StratifiedKFold

# Exact Kaggle Input Path
BASE_INPUT = '/kaggle/input/datasets/soumitrachowdhury003/hri30-70-10-20-split/HRI30_70_10_20'
stgcn_dir = f'{BASE_INPUT}/stgcn_format'
ctrgcn_dir = f'{BASE_INPUT}/ctrgcn_format'

# Load all ST-GCN data (Train + Val + Test = 2940)
X_st_train = np.load(f'{stgcn_dir}/train_data.npy')
X_st_val = np.load(f'{stgcn_dir}/val_data.npy')
X_st_test = np.load(f'{stgcn_dir}/test_data.npy')
X_all_st = np.concatenate([X_st_train, X_st_val, X_st_test])

with open(f'{stgcn_dir}/train_label.pkl', 'rb') as f: _, y_train = pickle.load(f)
with open(f'{stgcn_dir}/val_label.pkl', 'rb') as f: _, y_val = pickle.load(f)
with open(f'{stgcn_dir}/test_label.pkl', 'rb') as f: _, y_test = pickle.load(f)
y_all = np.array(list(y_train) + list(y_val) + list(y_test))

# Load all CTR-GCN data
npz = np.load(f'{ctrgcn_dir}/HRI30_CS.npz')
X_all_ctr = np.concatenate([npz['x_train'], npz['x_val'], npz['x_test']])

print(f"Total ST-GCN samples: {X_all_st.shape}")
print(f"Total CTR-GCN samples: {X_all_ctr.shape}")
print(f"Total labels: {y_all.shape}")

# Create 5-Fold Splits
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_indices = list(skf.split(X_all_st, y_all))

with open('/kaggle/working/cv_indices.pkl', 'wb') as f:
    pickle.dump(fold_indices, f)

print("5-Fold splits created and saved to /kaggle/working/cv_indices.pkl")

Total ST-GCN samples: (2940, 3, 150, 25, 1)
Total CTR-GCN samples: (2940, 3, 150, 25, 1)
Total labels: (2940,)
5-Fold splits created and saved to /kaggle/working/cv_indices.pkl


In [ ]:
# 5-Fold Cross Validation Training (7.5 hours)
import time
import math
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

class HRI30Dataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

NUM_EPOCHS = 50
BATCH_SIZE = 32
start_time = time.time()

for fold, (train_idx, test_idx) in enumerate(fold_indices):
    fold_num = fold + 1
    print(f"\n{'='*60}")
    print(f"STARTING FOLD {fold_num}/5")
    print(f"{'='*60}")

    # --- 1. ST-GCN Training ---
    stgcn_ckpt_path = os.path.join(OUT_DIR, f'fold{fold_num}_stgcn_best.pt')
    if os.path.exists(stgcn_ckpt_path):
        print(f"Skipping ST-GCN for Fold {fold_num} (Checkpoint exists).")
    else:
        print(f"Training ST-GCN for Fold {fold_num}...")
        X_train_fold, y_train_fold = X_all_st[train_idx], y_all[train_idx]
        X_test_fold, y_test_fold = X_all_st[test_idx], y_all[test_idx]

        train_loader = DataLoader(HRI30Dataset(X_train_fold, y_train_fold), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        test_loader = DataLoader(HRI30Dataset(X_test_fold, y_test_fold), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        model = STGCN_Model(in_channels=3, num_class=60, dropout=0.5, edge_importance_weighting=True, graph_args={'layout': 'ntu-rgb+d', 'strategy': 'spatial'})
        model.load_state_dict(torch.load(f'{STGCN_DIR}/models/st_gcn.ntu-xsub.pt', map_location='cpu', weights_only=False))
        model.fcn = nn.Conv2d(256, 30, kernel_size=1)
        nn.init.normal_(model.fcn.weight, 0, math.sqrt(2./30))
        model = model.to(device)

        backbone_params = [p for n, p in model.named_parameters() if 'fcn' not in n]
        head_params = [p for n, p in model.named_parameters() if 'fcn' in n]
        optimizer = optim.SGD([{'params': backbone_params, 'lr': 1e-3}, {'params': head_params, 'lr': 1e-2}], momentum=0.9, nesterov=True, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40], gamma=0.1)
        criterion = nn.CrossEntropyLoss()

        best_acc = 0.0
        for epoch in range(1, NUM_EPOCHS + 1):
            model.train()
            for bx, by in train_loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                optimizer.step()
            scheduler.step()

            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for bx, by in test_loader:
                    bx = bx.to(device)
                    preds = model(bx).argmax(dim=1).cpu()
                    correct += (preds == by).sum().item()
                    total += by.size(0)
            acc = 100.0 * correct / total

            if acc > best_acc:
                best_acc = acc
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'test_acc': acc}, stgcn_ckpt_path)
            if epoch % 10 == 0:
                print(f"  ST Epoch {epoch}/{NUM_EPOCHS} Acc: {acc:.2f}% (Best: {best_acc:.2f}%)")
        print(f"ST-GCN Fold {fold_num} done. Best Acc: {best_acc:.2f}%")

    # --- 2. CTR-GCN Training ---
    ctrgcn_ckpt_path = os.path.join(OUT_DIR, f'fold{fold_num}_ctrgcn_best.pt')
    if os.path.exists(ctrgcn_ckpt_path):
        print(f"Skipping CTR-GCN for Fold {fold_num} (Checkpoint exists).")
    else:
        print(f"Training CTR-GCN for Fold {fold_num}...")
        X_train_fold, y_train_fold = X_all_ctr[train_idx], y_all[train_idx]
        X_test_fold, y_test_fold = X_all_ctr[test_idx], y_all[test_idx]

        train_loader = DataLoader(HRI30Dataset(X_train_fold, y_train_fold), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        test_loader = DataLoader(HRI30Dataset(X_test_fold, y_test_fold), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        model = CTRGCN_Model(num_class=30, num_point=25, num_person=1, graph='graph.ntu_rgb_d.Graph', graph_args={'labeling_mode': 'spatial'})

        pretrained_dict = torch.load(f'{CTRGCN_DIR}/pretrained_model/NTU60_Xsub/CTRGCN_joint_89.9/runs-60-37560.pt', map_location='cpu', weights_only=False)
        if 'model_state_dict' in pretrained_dict:
            pretrained_dict = pretrained_dict['model_state_dict']
        pretrained_dict = {k[7:] if k.startswith('module.') else k: v for k, v in pretrained_dict.items()}
        model_dict = model.state_dict()
        filtered_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.shape == model_dict[k].shape}

        model.load_state_dict(filtered_dict, strict=False)
        model = model.to(device)

        backbone_params = [p for n, p in model.named_parameters() if 'fc' not in n]
        head_params = [p for n, p in model.named_parameters() if 'fc' in n]
        optimizer = optim.SGD([{'params': backbone_params, 'lr': 1e-3}, {'params': head_params, 'lr': 1e-2}], momentum=0.9, weight_decay=0.0004, nesterov=True)
        scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40], gamma=0.1)
        criterion = nn.CrossEntropyLoss()

        best_acc = 0.0
        for epoch in range(1, NUM_EPOCHS + 1):
            model.train()
            for bx, by in train_loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                optimizer.step()
            scheduler.step()

            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for bx, by in test_loader:
                    bx = bx.to(device)
                    preds = model(bx).argmax(dim=1).cpu()
                    correct += (preds == by).sum().item()
                    total += by.size(0)
            acc = 100.0 * correct / total

            if acc > best_acc:
                best_acc = acc
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'test_acc': acc}, ctrgcn_ckpt_path)
            if epoch % 10 == 0:
                print(f"  CTR Epoch {epoch}/{NUM_EPOCHS} Acc: {acc:.2f}% (Best: {best_acc:.2f}%)")
        print(f"CTR-GCN Fold {fold_num} done. Best Acc: {best_acc:.2f}%")

    # --- ETA Calculation ---
    elapsed = time.time() - start_time
    avg_time = elapsed / fold_num
    remaining = avg_time * (5 - fold_num)
    print(f"\nFold {fold_num} complete. Estimated time remaining: {remaining/3600:.2f} hours")

print("\nAll 5 folds training complete!")


STARTING FOLD 1/5
Training ST-GCN for Fold 1...
  ST Epoch 10/50 Acc: 39.29% (Best: 41.33%)
  ST Epoch 20/50 Acc: 51.87% (Best: 54.08%)
  ST Epoch 30/50 Acc: 58.50% (Best: 58.50%)
  ST Epoch 40/50 Acc: 60.20% (Best: 60.37%)
  ST Epoch 50/50 Acc: 61.73% (Best: 61.73%)
ST-GCN Fold 1 done. Best Acc: 61.73%
Training CTR-GCN for Fold 1...
  CTR Epoch 10/50 Acc: 65.82% (Best: 65.82%)
  CTR Epoch 20/50 Acc: 67.69% (Best: 68.03%)
  CTR Epoch 30/50 Acc: 67.86% (Best: 68.88%)
  CTR Epoch 40/50 Acc: 67.69% (Best: 68.88%)
  CTR Epoch 50/50 Acc: 67.69% (Best: 70.41%)
CTR-GCN Fold 1 done. Best Acc: 70.41%

Fold 1 complete. Estimated time remaining: 5.05 hours

STARTING FOLD 2/5
Training ST-GCN for Fold 2...
  ST Epoch 10/50 Acc: 45.92% (Best: 45.92%)
  ST Epoch 20/50 Acc: 55.44% (Best: 55.44%)
  ST Epoch 30/50 Acc: 61.56% (Best: 61.56%)
  ST Epoch 40/50 Acc: 62.07% (Best: 62.07%)
  ST Epoch 50/50 Acc: 62.59% (Best: 62.93%)
ST-GCN Fold 2 done. Best Acc: 62.93%
Training CTR-GCN for Fold 2...
  CTR Ep

In [ ]:
# Evaluate Consensus Layer on 5-Fold Test Sets
import csv
import pickle
import torch.nn as nn

if 'fold_indices' not in dir():
    with open('/kaggle/working/cv_indices.pkl', 'rb') as f:
        fold_indices = pickle.load(f)

def get_model_logits(model, data_np, batch_size=64):
    model.eval()
    all_logits = []
    N = data_np.shape[0]
    with torch.no_grad():
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = torch.tensor(data_np[start:end], dtype=torch.float32).to(device)
            output = model(batch)
            all_logits.append(output.cpu().numpy())
    return np.concatenate(all_logits)

def softmax(logits):
    exp_x = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def entropy(probs):
    return -np.sum(probs * np.log(probs + 1e-9), axis=1)

def consensus_fusion(logits_st, logits_ctr):
    probs_st = softmax(logits_st)
    probs_ctr = softmax(logits_ctr)
    H_st = entropy(probs_st)
    H_ctr = entropy(probs_ctr)
    W_st = np.exp(-H_st)
    W_ctr = np.exp(-H_ctr)
    sum_W = W_st + W_ctr
    W_st = W_st / sum_W
    W_ctr = W_ctr / sum_W
    probs_fused = W_st[:, None] * probs_st + W_ctr[:, None] * probs_ctr
    return np.argmax(probs_fused, axis=1)

OUT_DIR = '/kaggle/working'
csv_path = os.path.join(OUT_DIR, 'cv_results.csv')

with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Fold', 'ST-GCN_Acc', 'CTR-GCN_Acc', 'Consensus_Acc'])

    for fold, (train_idx, test_idx) in enumerate(fold_indices):
        fold_num = fold + 1
        print(f"\nEvaluating Fold {fold_num}...")

        # Load best ST-GCN model for this fold
        stgcn_model = STGCN_Model(in_channels=3, num_class=60, dropout=0.5, edge_importance_weighting=True, graph_args={'layout': 'ntu-rgb+d', 'strategy': 'spatial'})
        stgcn_model.fcn = nn.Conv2d(256, 30, kernel_size=1) # Mirror Cell 3 setup
        stgcn_ckpt = torch.load(os.path.join(OUT_DIR, f'fold{fold_num}_stgcn_best.pt'), map_location='cpu', weights_only=False)
        stgcn_model.load_state_dict(stgcn_ckpt['model_state_dict'])
        stgcn_model = stgcn_model.to(device).eval()

        # Load best CTR-GCN model for this fold
        ctrgcn_model = CTRGCN_Model(num_class=30, num_point=25, num_person=1, graph='graph.ntu_rgb_d.Graph', graph_args={'labeling_mode': 'spatial'})
        ctrgcn_ckpt = torch.load(os.path.join(OUT_DIR, f'fold{fold_num}_ctrgcn_best.pt'), map_location='cpu', weights_only=False)
        ctrgcn_state = ctrgcn_ckpt['model_state_dict']
        ctrgcn_state = {k[7:] if k.startswith('module.') else k: v for k, v in ctrgcn_state.items()}
        ctrgcn_model.load_state_dict(ctrgcn_state, strict=True)
        ctrgcn_model = ctrgcn_model.to(device).eval()

        # Get test data for this fold
        X_test_st = X_all_st[test_idx]
        X_test_ctr = X_all_ctr[test_idx]
        y_test = y_all[test_idx]

        # Run inference
        logits_st = get_model_logits(stgcn_model, X_test_st)
        logits_ctr = get_model_logits(ctrgcn_model, X_test_ctr)

        st_preds = np.argmax(logits_st, axis=1)
        ctr_preds = np.argmax(logits_ctr, axis=1)
        con_preds = consensus_fusion(logits_st, logits_ctr)

        st_acc = 100.0 * (st_preds == y_test).mean()
        ctr_acc = 100.0 * (ctr_preds == y_test).mean()
        con_acc = 100.0 * (con_preds == y_test).mean()

        print(f"  ST-GCN: {st_acc:.2f}% | CTR-GCN: {ctr_acc:.2f}% | Consensus: {con_acc:.2f}%")
        writer.writerow([fold_num, f"{st_acc:.2f}", f"{ctr_acc:.2f}", f"{con_acc:.2f}"])

print(f"\nResults saved to {csv_path}")
print("M6 Cross-Validation Complete!")


Evaluating Fold 1...
  ST-GCN: 61.73% | CTR-GCN: 70.41% | Consensus: 69.73%

Evaluating Fold 2...
  ST-GCN: 62.93% | CTR-GCN: 66.67% | Consensus: 67.69%

Evaluating Fold 3...
  ST-GCN: 53.40% | CTR-GCN: 69.05% | Consensus: 68.54%

Evaluating Fold 4...
  ST-GCN: 59.52% | CTR-GCN: 68.20% | Consensus: 69.22%

Evaluating Fold 5...
  ST-GCN: 56.29% | CTR-GCN: 67.86% | Consensus: 68.20%

Results saved to /kaggle/working/cv_results.csv
M6 Cross-Validation Complete!
